Analise de Levantamento de Pavimentação

In [53]:
!pip freeze > "requirements.txt"

Importação de Bibliotecas

In [54]:
import pandas as pd
import requests as rq
import glob
import os
import re

Obtendo arquivos da Fonte de Dados Oficial do DNIT

In [55]:
# ============================================================
# CONFIGURAÇÃO
# ============================================================

diretorio = "dados/raw/levantamentos_pavimentada"
os.makedirs(diretorio, exist_ok=True)

dataset_id = "b41e3382-79f5-4b1d-b936-dc75aaa82dfa"

# API do catálogo de dados
api_url = (
    "https://servicos.dnit.gov.br/dadosabertos/api/3/action/package_show"
    f"?id={dataset_id}"
)

# ============================================================
# 1. CONSULTAR OS RECURSOS DISPONÍVEIS
# ============================================================

response = rq.get(api_url)
response.raise_for_status()

dataset = response.json()["result"]

recursos = dataset["resources"]

# ============================================================
# 2. FILTRAR OS CSVs DE LEVANTAMENTOS_PAVIMENTADA
# ============================================================

arquivos = []

for recurso in recursos:

    nome = recurso.get("name", "")
    url = recurso.get("url", "")

    # Procura arquivos no padrão:
    # levantamentos_pavimentada_2026_08.csv

    match = re.search(
        r"levantamentos_pavimentada_(\d{4})_(\d{2})\.csv",
        nome.lower()
    )

    if not match:
        match = re.search(
            r"levantamentos_pavimentada_(\d{4})_(\d{2})\.csv",
            url.lower()
        )

    if match:
        ano = int(match.group(1))
        mes = int(match.group(2))

        arquivos.append({
            "ano": ano,
            "mes": mes,
            "nome": nome,
            "url": url
        })

# ============================================================
# 3. ORDENAR DO MAIS RECENTE PARA O MAIS ANTIGO
# ============================================================

arquivos = sorted(
    arquivos,
    key=lambda x: (x["ano"], x["mes"]),
    reverse=True
)

# Pegar os 6 meses mais recentes
ultimos_6 = arquivos[:6]

print("Arquivos selecionados:")
for arquivo in ultimos_6:
    print(
        f"{arquivo['ano']}-{arquivo['mes']:02d} "
        f"-> {arquivo['nome']}"
    )

# ============================================================
# 4. BAIXAR OS ARQUIVOS
# ============================================================

dfs = []

for arquivo in ultimos_6:

    nome_arquivo = (
        f"levantamentos_pavimentada_"
        f"{arquivo['ano']}_{arquivo['mes']:02d}.csv"
    )

    caminho = os.path.join(
        diretorio,
        nome_arquivo
    )

    print(f"\nBaixando: {nome_arquivo}")

    r = rq.get(arquivo["url"])
    r.raise_for_status()

    with open(caminho, "wb") as f:
        f.write(r.content)

    # Ler CSV
    df = pd.read_csv(
        caminho,
        sep=";"
    )

    # Identificar a competência
    df["ano"] = arquivo["ano"]
    df["mes"] = arquivo["mes"]

    # Guardar origem
    df["arquivo_origem"] = nome_arquivo

    dfs.append(df)

# ============================================================
# 5. CONSOLIDAR OS 6 MESES
# ============================================================

df_combined = pd.concat(
    dfs,
    ignore_index=True
)

# ============================================================
# 6. RESULTADO
# ============================================================

print("\n===================================")
print("DOWNLOAD CONCLUÍDO")
print("===================================")

print(f"Arquivos baixados: {len(dfs)}")
print(f"Linhas: {len(df_combined):,}")
print(f"Colunas: {len(df_combined.columns)}")

print("\nDimensão:")
print(df_combined.shape)

print("\nPrimeiras linhas:")
print(df_combined.head())


Arquivos selecionados:
2026-08 -> Condições Do Pavimento Levantamentos Agosto/2026
2026-07 -> Condições Do Pavimento Levantamentos Julho/2026
2026-06 -> Condições Do Pavimento Levantamentos Junho/2026
2026-05 -> Condições Do Pavimento Levantamentos Maio/2026
2026-02 -> Condições Do Pavimento Levantamentos Fevereiro/2026
2026-01 -> Condições do Pavimento Levantamentos Janeiro/2026

Baixando: levantamentos_pavimentada_2026_08.csv

Baixando: levantamentos_pavimentada_2026_07.csv

Baixando: levantamentos_pavimentada_2026_06.csv

Baixando: levantamentos_pavimentada_2026_05.csv

Baixando: levantamentos_pavimentada_2026_02.csv

Baixando: levantamentos_pavimentada_2026_01.csv

DOWNLOAD CONCLUÍDO
Arquivos baixados: 6
Linhas: 944,868
Colunas: 27

Dimensão:
(944868, 27)

Primeiras linhas:
   id_malha  UF         Contrato   Ano  Mes   Panela Remendo Trincamento  \
0     19110  RR  "26 00650/2025"  2026    8      Bom     Bom         Bom   
1     19110  RR  "26 00650/2025"  2026    8      Bom     Bo

In [56]:
path_list=glob.glob(f"dados/raw/levantamentos_pavimentada/*.csv")

dfs=[]
for path in path_list:
    df=pd.read_csv(path,sep=";")
    df["arquivo"]=path
    dfs.append(df)

df_combined = pd.concat(dfs,ignore_index=True)

df_combined

,id_malha,UF,Contrato,Ano,Mes,Rodovia,km,Sentido,Km_Inicial,Km_Final,...,ICM,ICM_Unificado,arquivo,Panela,Remendo,Trincamento,Rocada,Drenagem,Sinalizacao_Vertical,Sinalizacao_Horizontal
0,17475,RR,"""26 00650/2025""",2026,1,BR-210,89.0,D,90.0,89.0,...,46.0,53.00,dados/raw/levantamentos_pavimentada\levantamen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,17475,RR,"""26 00650/2025""",2026,1,BR-210,89.0,C,89.0,90.0,...,53.0,53.00,dados/raw/levantamentos_pavimentada\levantamen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,17475,RR,"""26 00650/2025""",2026,1,BR-210,90.0,D,91.0,90.0,...,73.0,73.00,dados/raw/levantamentos_pavimentada\levantamen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,17475,RR,"""26 00650/2025""",2026,1,BR-210,90.0,C,90.0,91.0,...,51.25,73.00,dados/raw/levantamentos_pavimentada\levantamen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,17475,RR,"""26 00650/2025""",2026,1,BR-210,91.0,C,91.0,92.0,...,51.25,51.25,dados/raw/levantamentos_pavimentada\levantamen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
944863,19623,MT,"""00 00003/2026""",2026,8,BR-364,1257.0,C,1257.0,1258.0,...,"6,75",6.75,dados/raw/levantamentos_pavimentada\levantamen...,Bom,Bom,Bom,Bom,Ruim,Bom,Ruim
944864,19623,MT,"""00 00003/2026""",2026,8,BR-364,1258.0,C,1258.0,1259.0,...,"6,75",10.25,dados/raw/levantamentos_pavimentada\levantamen...,Bom,Bom,Bom,Bom,Ruim,Bom,Ruim
944865,19623,MT,"""00 00003/2026""",2026,8,BR-364,1258.0,D,1259.0,1258.0,...,"10,25",10.25,dados/raw/levantamentos_pavimentada\levantamen...,Bom,Bom,Regular,Bom,Ruim,Bom,Ruim
944866,19623,MT,"""00 00003/2026""",2026,8,BR-364,1259.0,C,1259.0,1260.0,...,"1,875",6.50,dados/raw/levantamentos_pavimentada\levantamen...,Bom,Bom,Bom,Bom,Bom,Bom,Regular
